# basic

In [1]:
%load_ext autoreload
%autoreload all

In [9]:
import polars as pl
import numpy as np
import pickle
from tqdm import tqdm
import src.configs as configs
from src.tokenizer import GraphTokenizer

# 0. load files

In [ ]:
with open(configs.ProcessedGraph().candidate_is_a_reachable_dict, "rb") as f:
    candidate_reachable_child_map = pickle.load(f)

df_concept_hug_w_depth = pl.read_parquet(configs.ProcessedGraph.concept_w_depth)
mapped_candidate_rel_dist_prop = (pl.read_parquet(configs.ProcessedGraph().mapped_candidate_rel_dist_prop)
                                  
                                  # remove all relations in subgraph where candidate and mapped are distant
                                  .filter(pl.col("distance") <= configs.TokenizerParam().max_dist_candidate)
                                  )

# 1. tokenizer test (max dist  == 3)

In [38]:
tokenizer = GraphTokenizer(df_concept_hug_w_depth,
                        mapped_candidate_rel_dist_prop,
                        candidate_reachable_child_map,
                        configs.TokenizerParam().max_dist_candidate
                        )

# tryout

In [40]:
# top 8000 candidates of highest degree
example_candidates = mapped_candidate_rel_dist_prop.group_by("dst.id").agg(num_src = pl.col("src.id").n_unique()).sort(by = "num_src", descending=True).head(8000)["dst.id"].to_list()
scores, results, df_tok_all_n_dist = tokenizer.evaluate_components_and_tokenize(example_candidates, debug=True)

ComputeError: KeyboardInterrupt: 

In [28]:
scores

{'sem_cov_score': 0.8257357176029023,
 'distance_score': 0.9998577570503222,
 'uniqueness_entropy_score': 0.9632947235160737,
 'conciseness_score': 0.2121829324922644,
 'compression_rate': 0.1733477789815818,
 'UNK_rate': 0.0,
 'exact_rate': 0.040281690140845074}